# Extract launch tables from a historical Wikipedia revision

IBM Applied Data Science Capstone. Read the saved extraction offline by default. Set the refresh flag to request the historical revision and parse its flight rows again.

In [ ]:
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup
refresh_from_wikipedia = False
url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'

In [ ]:
def parse_launches(html):
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    def text(cell):
        return cell.get_text(' ', strip=True)
    for row in soup.select('table.wikitable.plainrowheaders.collapsible tr'):
        header = row.find('th', recursive=False)
        fields = row.find_all('td', recursive=False)
        if not header or not text(header).isdigit() or len(fields) < 9:
            continue
        moment = list(fields[0].stripped_strings)
        rows.append({'Flight No.': int(text(header)), 'Launch site': text(fields[2]),
                     'Payload': text(fields[3]), 'Payload mass': text(fields[4]),
                     'Orbit': text(fields[5]), 'Customer': text(fields[6]),
                     'Launch outcome': text(fields[7]), 'Version Booster': text(fields[1]),
                     'Booster landing': text(fields[8]), 'Date': moment[0].rstrip(','),
                     'Time': next((part for part in moment[1:] if re.search(r'\d{2}:\d{2}',part)), None)})
    if not rows:
        raise ValueError('No launch rows found in the historical HTML tables')
    return pd.DataFrame(rows)

In [ ]:
if refresh_from_wikipedia:
    response = requests.get(url, timeout=30, headers={'User-Agent':'IBM-capstone-coursework/1.0'})
    response.raise_for_status()
    launches = parse_launches(response.content)
else:
    launches = pd.read_csv('data/spacex_web_scraped.csv')
launches[['Flight No.','Date','Launch site','Launch outcome']].head()

In [ ]:
launches.groupby('Orbit', dropna=False).size().sort_values(ascending=False).head(8)

## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


Copyright © 2021 IBM Corporation. All rights reserved.
